### Imports

In [6]:

import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path


from pyNanoMatBuilder import crystalNPs as cyNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib

from ase import io
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase.io import read
from ase.io import write
from ase.visualize import view

from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup

import visualID as vID
from visualID import  fg, hl, bg

import numpy as np


/home/sara/pyNanoMatBuilder


### Functions that are defined in utils 

In [2]:
# in utils


#######################################################################
######################################## cif files informations
def get_crystal_type(self):
    """
    Find the Bravais lattice based on the space group number.
    """  
    spacegroup_number = self.ucSG.no  # space group number
    
    # bravais lattice based on space group number https://fr.wikipedia.org/wiki/Groupe_d%27espace
    if 195 <= spacegroup_number <= 230:  # Cubic
        if spacegroup_number == 225:
            return 'fcc'
        elif spacegroup_number == 229:
            return 'bcc'
        else:
            return 'cubic'
    elif 168 <= spacegroup_number <= 194:  # Hexagonal
        return 'hcp'
    elif 75 <= spacegroup_number <= 142:  # Tetragonal
        return 'tetragonal'
    elif 16 <= spacegroup_number <= 74:  # Orthorhombic
        return 'orthorhombic'
    elif 3 <= spacegroup_number <= 15:  # Monoclinic
        return 'monoclinic'
    elif 1 <= spacegroup_number <= 2:  # Triclinic
        return 'triclinic'
    else:
        return 'unknown'
   
def FindInterAtomicDist(self) :
    """
    Extract the interatomic distance based on the Bravais lattice.
    """   
    import math
    if self.crystal_type=='fcc':
        d=self.parameters[0]*math.sqrt(2)/2
    
    if self.crystal_type=='bcc' :
        d=self.parameters[0]*math.sqrt(3)/2
    
    if self.crystal_type=='hcp' :
        d_a=self.parameters[0]
        d_c=self.parameters[2]/2
        if d_a>d_c: #if compact
            d=d_c
        if d_c>d_a: #if not compact
            d=d_a

    return d

def extract_cif_info(self,cif_file):
    """
    Extract useful information from a CIF file, including:
    - Structure loaded with ASE.
    - Bravais lattice system.
    - Chemical name(s).
    """   
    structure = read(cif_file) # load structure with ase
    # self.ucUnitcell = self.cif.cell.cellpar()
    # self.ucV = cellpar_to_cell(self.ucUnitcell)
    self.parameters=self.cif.cell.lengths()
    #print('cristal parameters', self.parameters)
    # self.ucBL = self.cif.cell.get_bravais_lattice()
    self.ucSG = get_spacegroup(self.cif,symprec= float(1e-2))
    self.ucFormula = self.cif.get_chemical_formula()
    self.crystal_type = get_crystal_type(self)
    return {
        # 'structure': self.ucUnitcell,
        # 'lattice_system':self.ucBL,
        # 'crystal_name': self.ucFormula,
        'cif_path': cif_file,
        'crystal_type' : self.crystal_type
        
    }
        
def load_cif(self, cif_file,noOutput):
    """
    Load a CIF file.
    """      
    cif_folder = "cif_database"
    path2cif = Path(os.path.join(cif_folder, cif_file)).resolve()
    self.cif = io.read(path2cif)
    if not noOutput :
        print("Absolute path to CIF:", path2cif)
    if not path2cif.exists():
        raise FileNotFoundError(f"File {cif_file} not found.")
    if path2cif not in self.loaded_cifs:
        self.loaded_cifs[path2cif] = extract_cif_info(self,path2cif)
    return self.loaded_cifs[path2cif]




def writexyz_generalized_crystals(self,structure,path,instance_class_crystals, number,noOutput:bool=True):
    '''
    simple xyz writing, with atomic symbols/x/y/z understandable by DebyeCalculator with a dictionnary containing the main informations of the nps
    input : 
            - path : path in which the xyz files are created
            - instance_class_crystals: instance of the crystal class
            - number : defined in the call of the function

    Note : Dimensions are in Å, MOI are in amu.Å², normalized MOI in Å²
    '''
    # verify if the path does exist
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Directory '{path}'does not exist.")
    
    # extract attributes from the class to write the dictionnary and the name of the file
    if isinstance(instance_class_crystals,object):
        element=self.cif_name.split()[0]
        if structure== None : #example for NaCl, get the lattice for the name of the file
            structure=self.crystal_type
        
        crystalStructure=self.crystal_type
        number2=0
        metadata = instance_class_crystals.__dict__.copy() # Get all attributes
        shape= instance_class_crystals.shape
        # 1) is it a wulff shape ? 2) if it's a wire : get nRot (number of edges of the cross section) and refplanewire (plane rotated to create the wire) 
        nRot=None
        plane_rotated=None
        if "Wulff" in shape: 
            shape=shape.split(':')[1]
            wulff= True
            if "hcpwire" in shape :
                nRot=int(6)
                plane_rotated=np.array(instance_class_crystals.surfacesWulff)
                
        elif "wire" in shape :
            wulff= False
            nRot=instance_class_crystals.nRotWire
            plane_rotated=instance_class_crystals.refPlaneWire
        else : 
            wulff= False
     
        NP=instance_class_crystals.NP
        element_array=NP.get_chemical_symbols()
        MOI=np.round(instance_class_crystals.moi,3) #amu.angs²
        MOI_normalized=np.round(instance_class_crystals.moisize,3)  #angs²
        dim_main=np.round(np.array(instance_class_crystals.dim),3) #angs
        number_atoms=int(instance_class_crystals.nAtoms)
        radius1=round(instance_class_crystals.radiusCircumscribedSphere,3) #angs
        diam1=round(radius1*2*0.1,2)
        radius2=round(instance_class_crystals.radiusInscribedSphere,3) #angs
        composition={}
        
        for elements in element_array:
            if elements in composition:
                composition[elements]+=1
            else:
                composition[elements]=1

        coord=NP.get_positions()
        natoms=len(element_array) 
        # write the xyz file and cif file
        # filename_xyz= f"{element}_{structure}_{shape}_{'{:07d}'.format(number)}_{'{:07d}'.format(number2)}.xyz"
        filename_xyz= f"{element}_{structure}_{shape}_{diam1}.xyz"
        full_path = os.path.join(path, filename_xyz)
        if not noOutput :
            print('xyz file created:',full_path)
        line2write='%d \n'%natoms
   
        dictionnary = {
            'composition': composition,
            'crystal structure': crystalStructure,
            'shape': shape,
            'MOI': MOI,
            'MOInormalized': MOI_normalized,
            'main_dimensions': dim_main,
            'inscribed_sphere_radius': radius2,
            'circumscribed_sphere_radius': radius1,
            'number_of_atoms': number_atoms,
            'wulff': wulff,
            'crystallization': {
                'state': 'crystalline',
                'type': 'monocrystalline',
                'subtype': 'random'
            },
            'wire_description': {
                'nRot': nRot,
                'plane_rotated': plane_rotated
            }
            }

        
        
        line2write+='%s \n'%dictionnary
        # writing of the coordinates
        for i in range(natoms):
            line2write+='%s'%str(element_array[i])+'\t %.8f'%float(coord[i,0])+'\t %.8f'%float(coord[i,1])+'\t %.8f'%float(coord[i,2])+'\n'
        with open(full_path,'w') as file:
            file.write(line2write)
        
        # write the cif files using the function write from ASE.io
        filename_cif = filename_xyz.replace('.xyz', '.cif')
        new_path = os.path.join(path, filename_cif)
        if not noOutput :
            print('cif file created:',new_path)
        write(new_path, instance_class_crystals.NP)     
  
    else :
        print('Objet is not a class instance')


### Class PredifinedWulffFiles_onecif : takes one cif as entry and creates the xyz/cif files for different Wulff forms and sizes

In [3]:
class PredifinedWulffFiles_onecif :
    
    def __init__(self, path, cif_file, wulff_shapes,sizes,form: str=None,noOutput:bool = True):
        """
        Initialize the class with CIF data, Wulff shapes information and size.
            Args:
        path (str) : path that will contain the created xyz files
        cif file (file): singular cif file
        wulff_shapes (dataframe): the wulff shapes and their informations
        sizes (array-like): array of the sizes of the nanoparticles
        form (str): if str=None : all the wulff forms are created but possiblity to pick a specific form
        noOutput (bool): if bool=False : details of the files
        
        """
        self.cif_file = cif_file  # cif file
        self.wulff_shapes = wulff_shapes  # DataFrame of Wulff  forms
        self.loaded_cifs = {}  # stock loaded cif files
        self.sizes= [[k] for k in sizes] #nested list of the sizes 
        self.form= form
        self.cif_name='Fe beta'
        # number_created_files=0
        self.create_wulff_shapes(cif_file,noOutput,path)
  

    def create_wulff_shapes(self,cif_file,noOutput,path):
        """
        Generate Wulff shapes for all valid combinations of CIF files and Wulff forms.
        """
        
        structure=None
        cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
        print('self.crystal_type',self.crystal_type)
        if self.form == None : #if the form isn't specific : 
            for form, row in self.wulff_shapes.iterrows():
                lattices = [l.strip() for l in row['Bravais lattice'].split(',')]
                if self.crystal_type  in lattices:  #verify if the lattice of the cif structure matches the lattice of the wulff form
                    index=0 
                    if not noOutput :
                        print(f" {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {form} form. \033[0m ")
                    # instance of the crystal class for each cif file and possible form and size
                    for i in self.sizes :
                        index += 1 
                        TestNP = cyNP.Crystal(
                            crystal=f'{self.cif_name}',
                            userDefCif=cif_info['cif_path'],
                            shape=f"Wulff: {form}",
                            sizesWulff= i,
                            threshold=0.001,
                            thresholdCoreSurface=2,
                            postAnalyzis=True,
                            jmolCrystalShape=True,
                            noOutput=True,
                            aseView=False,
                            skipSymmetryAnalyzis=True
                        ) 
 
                        pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)         
                else:
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {form} form .\033[0m  ")     
                        
       
        else:  #if a specific form is given by the user
               
            lattices = self.wulff_shapes['Bravais lattice'][f'{self.form}']
            if self.crystal_type in lattices:
                index=0 
                if not noOutput :
                    print(f" {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {self.form} form.\033[0m ")
                # instance of the crystal class for each cif file and possible form and size
                for i in self.sizes :
                    index += 1 
                    TestNP = cyNP.Crystal(
                        crystal=f'{self.cif_name}',
                        userDefCif=cif_info['cif_path'],
                        shape=f"Wulff: {self.form}",
                        sizesWulff= i,
                        threshold=0.001,
                        thresholdCoreSurface=2,
                        postAnalyzis=True,
                        jmolCrystalShape=True,
                        noOutput=True,
                        aseView=False,
                        skipSymmetryAnalyzis=True
                    ) 
                    pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)
                    # number_created_files+=1
     
            else:
                if not noOutput :
                    print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {self.form} form .\033[0m ")    


### Use of the class PredifinedWulffFiles_onecif

#### J'ai rentré les formes à la main à chaque fois (trOh, Oh, cubo, trcube, cube) pour pouvoir changer les bornes de taille (car elles ne définissent pas le diamètre circonscrit donc il faut tester à chaque fois)

In [4]:
t = pyNMBu.timer()
t.chrono_start()

path='/home/sara/pyNanoMatBuilder/Febeta_Nicolas'

sizes=np.arange(0.8,2,0.2) # sizes
class_test=PredifinedWulffFiles_onecif(path,cif_file='cod1539039-Fe_beta.cif', wulff_shapes=data.WulffShapes.WSdf,sizes=sizes,form='trcube',noOutput= False)
print(t.chrono_stop(hdelay=True))

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1539039-Fe_beta.cif
self.crystal_type cubic
  cubic corresponds to the lattices fcc, cubic of the Wulff trcube form. 
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.06.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.06.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.38.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.38.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.67.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.67.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.91.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_1.91.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ trcube_2.21.xyz
cif file created: /home/sara/pyNa

#### Sinon pas besoin de préciser les formes et le code vérifie lui même si la forme peut être créée selon le composé et sa strucure, et donc pour Fe_beta c'est bien trOh, Oh, cubo, trcube, cube, par contre les tailles sont définies de la même manière et pas possible de gérer les diamètres des sphères circonscrites

In [5]:
t = pyNMBu.timer()
t.chrono_start()

path='/home/sara/pyNanoMatBuilder/Febeta_Nicolas'

sizes=np.arange(0.8,2,0.2) # sizes
class_test=PredifinedWulffFiles_onecif(path,cif_file='cod1539039-Fe_beta.cif', wulff_shapes=data.WulffShapes.WSdf,sizes=sizes,noOutput= False)
print(t.chrono_stop(hdelay=True))

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1539039-Fe_beta.cif
self.crystal_type cubic
  cubic corresponds to the lattices ['bcc', 'fcc', 'cubic'] of the Wulff cube form.  
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_1.06.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_1.06.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_1.49.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_1.49.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_2.02.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_2.02.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_2.29.xyz
cif file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_2.29.cif
xyz file created: /home/sara/pyNanoMatBuilder/Febeta_Nicolas/Fe_cubic_ cube_2.34.xyz
cif file created: /home/sara/pyNanoMatB